In [6]:
#A.Struktur Direktori HDFS

!hdfs dfs -mkdir -p /user/mahasiswa/ecommerce/raw
!hdfs dfs -mkdir -p /user/mahasiswa/ecommerce/processed

!hdfs dfs -ls -R /user/mahasiswa/ecommerce

drwxr-xr-x   - aufaa supergroup          0 2026-09-09 06:16 /user/mahasiswa/ecommerce/processed
-rw-r--r--   1 aufaa supergroup      41257 2026-09-09 06:16 /user/mahasiswa/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 aufaa supergroup        530 2026-09-09 06:16 /user/mahasiswa/ecommerce/processed/ringkasan_kota_kategori.csv
drwxr-xr-x   - aufaa supergroup          0 2026-09-09 06:15 /user/mahasiswa/ecommerce/raw
-rw-r--r--   1 aufaa supergroup      12329 2026-09-09 06:15 /user/mahasiswa/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 aufaa supergroup      12154 2026-09-09 06:15 /user/mahasiswa/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 aufaa supergroup      12684 2026-09-09 06:15 /user/mahasiswa/ecommerce/raw/transaksi_yogyakarta.csv


In [7]:
#B.Upload Dataset ke HDFS

!hdfs dfs -put transaksi_magelang.csv /user/mahasiswa/ecommerce/raw/
!hdfs dfs -put transaksi_yogyakarta.csv /user/mahasiswa/ecommerce/raw/
!hdfs dfs -put transaksi_semarang.csv /user/mahasiswa/ecommerce/raw/

put: `/user/mahasiswa/ecommerce/raw/transaksi_magelang.csv': File exists
put: `/user/mahasiswa/ecommerce/raw/transaksi_yogyakarta.csv': File exists
put: `/user/mahasiswa/ecommerce/raw/transaksi_semarang.csv': File exists


In [8]:
!hdfs dfs -ls -h /user/mahasiswa/ecommerce/raw

Found 3 items
-rw-r--r--   1 aufaa supergroup     12.0 K 2026-09-09 06:15 /user/mahasiswa/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 aufaa supergroup     11.9 K 2026-09-09 06:15 /user/mahasiswa/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 aufaa supergroup     12.4 K 2026-09-09 06:15 /user/mahasiswa/ecommerce/raw/transaksi_yogyakarta.csv


In [9]:
#C.Membaca dan Menggabungkan Data dari HDFS

import pandas as pd

df_magelang = pd.read_csv(
    "http://localhost:9870/webhdfs/v1/user/mahasiswa/ecommerce/raw/transaksi_magelang.csv?op=OPEN"
)

df_yogyakarta = pd.read_csv(
    "http://localhost:9870/webhdfs/v1/user/mahasiswa/ecommerce/raw/transaksi_yogyakarta.csv?op=OPEN"
)

df_semarang = pd.read_csv(
    "http://localhost:9870/webhdfs/v1/user/mahasiswa/ecommerce/raw/transaksi_semarang.csv?op=OPEN"
)

df_gabungan = pd.concat(
    [df_magelang, df_yogyakarta, df_semarang],
    ignore_index=True
)

df_gabungan.head()

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang


In [10]:
df_gabungan["kota"].value_counts()

kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64

In [11]:
#D.Mengolah Data Transaksi

df_gabungan["total_pendapatan"] = (
    df_gabungan["unit_terjual"] *
    df_gabungan["harga_satuan"]
)

df_gabungan.head()

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota,total_pendapatan
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang,50000
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang,175000
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang,175000
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang,150000
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang,700000


In [12]:
ringkasan = (
    df_gabungan
    .groupby(["kota", "kategori"])["total_pendapatan"]
    .sum()
    .reset_index()
)

ringkasan

,kota,kategori,total_pendapatan
0,Magelang,Elektronik,18775000
1,Magelang,Fashion,27750000
2,Magelang,Kesehatan & Kecantikan,17375000
3,Magelang,Makanan & Minuman,17525000
4,Magelang,Rumah Tangga,12200000
5,Semarang,Elektronik,21425000
6,Semarang,Fashion,26425000
7,Semarang,Kesehatan & Kecantikan,13525000
8,Semarang,Makanan & Minuman,19750000
9,Semarang,Rumah Tangga,10975000


In [13]:
#Menyimpan Hasil Pengolahan

df_gabungan.to_csv(
    "data_gabungan_bersih.csv",
    index=False
)

In [14]:
ringkasan.to_csv(
    "ringkasan_kota_kategori.csv",
    index=False
)

In [15]:
!hdfs dfs -put data_gabungan_bersih.csv /user/mahasiswa/ecommerce/processed/
!hdfs dfs -put ringkasan_kota_kategori.csv /user/mahasiswa/ecommerce/processed/

put: `/user/mahasiswa/ecommerce/processed/data_gabungan_bersih.csv': File exists
put: `/user/mahasiswa/ecommerce/processed/ringkasan_kota_kategori.csv': File exists


In [16]:
!hdfs dfs -ls -h /user/mahasiswa/ecommerce/processed

Found 2 items
-rw-r--r--   1 aufaa supergroup     40.3 K 2026-09-09 06:16 /user/mahasiswa/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 aufaa supergroup        530 2026-09-09 06:16 /user/mahasiswa/ecommerce/processed/ringkasan_kota_kategori.csv


Refleksi

Pemisahan data raw dan processed pada HDFS memberikan keuntungan dalam pengelolaan data karena setiap jenis data memiliki tempat penyimpanan yang jelas. Data raw merupakan data asli yang belum mengalami perubahan sehingga dapat digunakan kembali apabila diperlukan proses pengolahan ulang. Sementara itu, data processed merupakan data yang sudah melalui proses pembersihan atau transformasi sehingga lebih siap digunakan untuk analisis.

Dengan memisahkan kedua jenis data tersebut, risiko tercampurnya data asli dengan data hasil pengolahan dapat dikurangi. Struktur folder juga menjadi lebih mudah dipahami sehingga proses pencarian data menjadi lebih cepat. Selain itu, apabila terjadi kesalahan dalam proses pengolahan, data raw masih tersedia sebagai sumber awal untuk melakukan proses ulang tanpa harus meminta atau membuat ulang dataset. Pemisahan ini juga membantu menjaga alur kerja data agar lebih terorganisir dan memudahkan proses pengembangan analisis Big Data pada tahap berikutnya.